In [105]:
# from pathlib import Path, PureWindowsPath # For windows only 
import os 
import pandas as pd
import time
from time import sleep
import selenium #Need this step 
from selenium import webdriver #Need this step 

from selenium.webdriver.common.by import By #Allows for selenium to click things 
from selenium.webdriver.chrome.service import Service #https://stackoverflow.com/questions/64717302/deprecationwarning-executable-path-has-been-deprecated-selenium-python
from selenium.webdriver.support import expected_conditions as EC #Allows for more complex code 
from selenium.webdriver.chrome.options import Options #Allows you to change aspects of the browser

# Establish options we can change
chrome_options = Options() 

In [264]:
chrome_options.add_argument("--window-size=1900,1000") #pick a size and stick with it, it can change how code intreacts with the website.
driver = webdriver.Chrome(options = chrome_options) #establish driver

In [265]:
url = 'https://lolalytics.com/lol/tierlist/'
driver.get(url) # Get the url
# Source - https://stackoverflow.com/a/27760083
# Posted by OWADVL, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-15, License - CC BY-SA 4.0

driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # scrolls to the very bottom of the page to load everything (takes ~20 seconds)
time.sleep(5) # want to wait 10 seconds to let things load
driver.execute_script("window.scrollBy(0, -6000);") # scrolls back up to load everything
for i in range(0,20): # scrolls downs for the next twenty seconds to make sure everything loads
    driver.execute_script("window.scrollBy(0, 300);")
    time.sleep(1)

In [209]:
buckets = driver.find_elements(By.XPATH, '/html/body/main/div[6]/div') # creating bucket of each thing
del buckets[:2]
buckets[160].text

'70\nWarwick\nD\n63.23\n49.12\n+0.58\n1.95\n1.05\n-2\n2,382\n166\n54.75\n3,978\n5.63\nD I'

In [182]:
buckets[171].text # testing

'92\nTrundle\nD\n64.18\n48.69\n+1.25\n1.13\n0.31\n-2\n1,378\n171\n54.39\n1,550\n5.70\nD I'

In [269]:
import time            # importing time package
start=time.time()      # start time

out = []
count = 3

for bucket in buckets:
    champion = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[3]/a')[0].text
    wr = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')[0].text
    
    temp= driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')
    wrdelta= temp[1].text if len(temp) > 1 else 0 # checks to see if there is an entry for wrdelta (some are missing on the website)

    pick = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[7]')[0].text
    ban = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[8]')[0].text
    PBI = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[9]')[0].text
    
    data = {
        'Champion':champion,
        'wr':wr,
        'wrdelta':wrdelta,
        'Pick Rate':pick,
        'Ban Rate':ban,
        'PBI Index':PBI,
    }
    out.append(data)
    count = count+1

end=time.time()        # end time

total_time=end-start   # measures total time by subtracting start by end
print(f' The code takes {round(total_time,5)} seconds to run.')

 The code takes 28.58022 seconds to run.


In [270]:
outdf=pd.DataFrame(out)
outdf

,Champion,wr,wrdelta,Pick Rate,Ban Rate,PBI Index
0,Lee Sin,50.64,+0.33,14.65,16.17,8
1,Kennen,53.09,+0.59,2.17,2.08,6
2,Jinx,52.36,+0.22,14.33,4.33,32
3,Rell,52.03,+0.36,3.48,1.07,6
4,Olaf,52.80,+0.66,2.40,2.26,6
...,...,...,...,...,...,...
167,Gnar,49.24,-0.14,4.54,1.76,-4
168,Renekton,49.49,+0.38,6.19,4.52,-5
169,Mel,44.82,+0.39,4.67,37.18,-40
170,Urgot,48.71,0,2.41,0.76,-4


In [274]:
outdf.to_csv('league.csv', index=False)